In [2]:
import os
from utils import *
from os.path import join
from pprint import pprint

import numpy as np
from scipy.stats import circmean
from math import radians

import requests
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
import os
import json
import model_helpers as mh
from neuron import h, load_mechanisms

--No graphics will be displayed.


In [4]:
code_version = 'Hay'
model_version = 'NeuroML'
nmldb_id = 'NMLCL000073'  # 'NMLCL000073' (Hay et al. 2011)
model_name = f'{nmldb_id}-{model_version}'
mh.download_from_nmldb(nmldb_id, model_version)

Model NMLCL000073 already downloaded.


In [5]:
sim_name = 'morphology'
cwd = os.getcwd()
models_dir = os.path.join(cwd, 'models') 
model_dir = os.path.join(models_dir, model_version, model_name)  # 'L5bPCmodelsEH')
hocs_dir = model_dir if 'biophys' not in model_name else os.path.join(model_dir,'models')
mod_dir = model_dir if 'biophys' not in model_name else os.path.join(model_dir, 'mod')
output_dir, sim_dir = mh.create_output_dirs(sim_name, model_dir)
output_dir, sim_dir

('/home/kedoxey/CRCNS/PyramidalCellSimulations/models/NeuroML/NMLCL000073-NeuroML/output',
 '/home/kedoxey/CRCNS/PyramidalCellSimulations/models/NeuroML/NMLCL000073-NeuroML/output/morphology')

In [6]:
cell_name = 'L5PC'
nml_file = os.path.join(model_dir,cell_name+'.cell.nml')
model_tree = ET.parse(nml_file)

In [7]:
root = model_tree.getroot()

In [8]:
root.tag, root.attrib

('{http://www.neuroml.org/schema/neuroml2}neuroml',
 {'{http://www.w3.org/2001/XMLSchema-instance}schemaLocation': 'http://www.neuroml.org/schema/neuroml2  https://raw.githubusercontent.com/NeuroML/NeuroML2/development/Schemas/NeuroML2/NeuroML_v2beta4.xsd',
  'id': 'L5PC'})

In [9]:
for child in root:
    print(child.tag, child.attrib)

{http://www.neuroml.org/schema/neuroml2}include {'href': 'Ca_HVA.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'Ca_LVAst.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'CaDynamics_E2_NML2__decay122__gamma5_09Emin4.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'CaDynamics_E2_NML2__decay460__gamma5_01Emin4.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'Ih.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'Im.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'K_Pst.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'K_Tst.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'Nap_Et2.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'NaTa_t.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'pas.channel.nml'}
{http://www.neuroml.org/schema/neuroml2}include {'href': 'SK_E2.channel.nml'}
{ht

In [10]:
cell= root[-1]
cell

<Element '{http://www.neuroml.org/schema/neuroml2}cell' at 0x7fe4a26dbb30>

In [11]:
for child in cell:
    print(child.tag, child.attrib)

{http://www.neuroml.org/schema/neuroml2}notes {}
{http://www.neuroml.org/schema/neuroml2}morphology {'id': 'morphology_L5PC'}
{http://www.neuroml.org/schema/neuroml2}biophysicalProperties {'id': 'biophys'}


## Morphology

In [12]:
morpho = cell[1]
morpho

<Element '{http://www.neuroml.org/schema/neuroml2}morphology' at 0x7fe4a26dba40>

In [13]:
segmentGroups = {}
for i, group in enumerate(morpho.findall('{http://www.neuroml.org/schema/neuroml2}segmentGroup')):
    
    segmentGroups.update({i:group})


In [23]:
"k8".isdigit()

False

### Segment names

In [27]:
seg_names = []
for i in segmentGroups:
    segmentGroup = segmentGroups[i]
    id = segmentGroup.attrib['id']

    if any(map(str.isdigit, id)):
        name = id.rsplit('_', 1)[0]
    else:
        name = id
    # name = id
    if name not in seg_names:
        seg_names.append(name)
seg_names

['soma',
 'axon',
 'apic',
 'dend',
 'all',
 'soma_group',
 'apical_dends_and_soma',
 'dends_and_soma',
 'soma_and_axon',
 'basal_dends_and_soma',
 'axon_group',
 'dendrite_group',
 'apical_dends',
 'non_hot_apical_dends',
 'OneSecGrp_SectionRef',
 'basal_dends']

In [29]:
comp_domain_names = ['soma_group', 'axon_group', 'apical_dends', 'basal_dends']
comp_domains = {}
for key, val in segmentGroups.items():
    if val.attrib['id'] in comp_domain_names:
        comp_domains.update({val.attrib['id']:val})

In [30]:
comp_domains

{'soma_group': <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7fe4a102e090>,
 'axon_group': <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7fe4a0fe8e00>,
 'apical_dends': <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7fe4a1006b80>,
 'basal_dends': <Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7fe4a0fa79a0>}

## Somatic compartments

In [31]:
comp_domains['soma_group']

<Element '{http://www.neuroml.org/schema/neuroml2}segmentGroup' at 0x7fe4a102e090>

In [32]:
for child in comp_domains['soma_group']:
    print(child.tag, child.attrib)

{http://www.neuroml.org/schema/neuroml2}include {'segmentGroup': 'soma_0'}


In [33]:
soma_group = comp_domains['soma_group'][0].attrib['segmentGroup']

In [34]:
soma_segments = {}
for segment in morpho.findall('{http://www.neuroml.org/schema/neuroml2}segment'):
    if soma_group in segment.attrib['name']:
        soma_segments.update({segment.attrib['id']:segment.attrib['name']})
soma_segments

{'0': 'Seg0_soma_0',
 '2': 'Seg1_soma_0',
 '3': 'Seg2_soma_0',
 '4': 'Seg3_soma_0',
 '5': 'Seg4_soma_0',
 '6': 'Seg5_soma_0',
 '7': 'Seg6_soma_0',
 '8': 'Seg7_soma_0',
 '9': 'Seg8_soma_0',
 '10': 'Seg9_soma_0',
 '11': 'Seg10_soma_0',
 '12': 'Seg11_soma_0',
 '13': 'Seg12_soma_0',
 '14': 'Seg13_soma_0',
 '15': 'Seg14_soma_0',
 '16': 'Seg15_soma_0',
 '17': 'Seg16_soma_0',
 '18': 'Seg17_soma_0',
 '19': 'Seg18_soma_0',
 '20': 'Seg19_soma_0'}

In [35]:
for child in morpho[0]:
    print(child.tag,child.attrib)

{http://www.neuroml.org/schema/neuroml2}proximal {'x': '34.1634', 'y': '17.6215', 'z': '-50.25', 'diameter': '3.80039'}
{http://www.neuroml.org/schema/neuroml2}distal {'x': '35.3196', 'y': '17.6937', 'z': '-50.25', 'diameter': '6.44373'}


In [53]:
these_segments = soma_segments
soma_length = 0
soma_parents = {}

for segment in morpho.findall('{http://www.neuroml.org/schema/neuroml2}segment'):
    seg_id = segment.attrib['id']
    
    if seg_id in these_segments.keys():
        
        if 'Seg0' in segment.attrib['name']:
            proximal = segment[0]
            distal = segment[1]

            soma_y_prox = float(proximal.attrib['y'])
            soma_y_dist = float(distal.attrib['y'])

            if soma_y_prox < soma_y_dist:
                lb = soma_y_prox
                ub = soma_y_dist
                soma_dir = 1  # prox->dist
                soma_root = lb
            else:
                lb = soma_y_dist
                ub = soma_y_prox
                soma_dir = -1  # dist->prox
                soma_root = ub

        else:
            parent_id = segment[0].attrib['segment']

            proximal = soma_parents[parent_id]
            distal = segment[1]

            if soma_dir==1 and float(distal.attrib['y'])>ub:
                ub = float(distal.attrib['y'])
            if soma_dir==1 and float(distal.attrib['y'])<lb:
                lb = float(distal.attrib['y'])  
        
        soma_parents.update({seg_id: distal})

        soma_length += compute_segment_length(proximal, distal)

soma_center = lb + (ub-lb)/2
soma_length = ub-lb

In [47]:
soma_center, lb, ub, soma_length, ub-lb

(18.34365, 17.6215, 19.0658, 1.4442999999999984, 1.4442999999999984)

## Stems

In [55]:
stems = {}

for segment in morpho.findall('{http://www.neuroml.org/schema/neuroml2}segment'):
    seg_id = segment.attrib['id']
    seg_name = segment.attrib['name']

    if not seg_name in soma_segments.values():

        parent_id = segment[0].attrib['segment']

        if parent_id in soma_parents.keys():
            stems.update({seg_id: seg_name})
    
print(f'Number of stems = {len(stems)}')
stems

Number of stems = 10


{'4067': 'Seg0_axon_0',
 '1660': 'Seg0_apic_0',
 '1594': 'Seg0_dend_79',
 '1585': 'Seg0_dend_78',
 '1260': 'Seg0_dend_63',
 '827': 'Seg0_dend_42',
 '749': 'Seg0_dend_39',
 '353': 'Seg0_dend_16',
 '206': 'Seg0_dend_7',
 '21': 'Seg0_dend_0'}

### compute stem cross-sectional areas

In [56]:
stem_names = []
stem_diams = []

for segment in morpho.findall('{http://www.neuroml.org/schema/neuroml2}segment'):
    seg_name = segment.attrib['name']

    if seg_name in stems.values():
                   
        proximal = segment[1]
        diameter = float(proximal.attrib['diameter'])
        stem_names.append(seg_name)
        stem_diams.append(diameter)

stem_diams = np.array(stem_diams)
stem_diams

array([ 1.  , 10.8 ,  0.88,  0.29,  1.46,  0.58,  0.88,  1.46,  0.29,
        0.58])

In [57]:
def compute_stem_csa(diam):
    return np.pi*np.square(diam/2)

In [59]:
stem_csas = compute_stem_csa(stem_diams)
stem_csas

array([7.85398163e-01, 9.16088418e+01, 6.08212338e-01, 6.60519855e-02,
       1.67415473e+00, 2.64207942e-01, 6.08212338e-01, 1.67415473e+00,
       6.60519855e-02, 2.64207942e-01])

In [60]:
stem_names

['Seg0_axon_0',
 'Seg0_apic_0',
 'Seg0_dend_79',
 'Seg0_dend_78',
 'Seg0_dend_63',
 'Seg0_dend_42',
 'Seg0_dend_39',
 'Seg0_dend_16',
 'Seg0_dend_7',
 'Seg0_dend_0']

In [61]:
np.sum(stem_csas)

97.61949392315906

In [63]:
for stem_id,name in stems.items():
    
    prox_list, dist_list = get_segment_coord_lists({stem_id:name},morpho,stems) 
    print(name,' angle:',compute_average_orientation(prox_list,dist_list)) # mean projection angle of all stem compartments off XZ-plane? -- needs to be restricted to local compartments e.g., apic


Seg0_axon_0  angle: 0.51678330799935
Seg0_apic_0  angle: 0.5771213449789857
Seg0_dend_79  angle: 0.34116452615574666
Seg0_dend_78  angle: 0.2050942633818013
Seg0_dend_63  angle: 0.1575069491200203
Seg0_dend_42  angle: 0.3198006361688046
Seg0_dend_39  angle: 0.2712484457812747
Seg0_dend_16  angle: 0.21440570451713192
Seg0_dend_7  angle: 0.24256382200546311
Seg0_dend_0  angle: 0.2818319164172626
